# Lab 12-02: Deep Research Loop

Runs an **agentic deep research loop** using `o3-deep-research` over the `arxiv-nlp`
Foundry IQ knowledge base created in Lab 10. The model iteratively calls `search` and
`fetch` tools backed by Foundry IQ, then `gpt-4.1-mini` synthesises a comprehensive
cited research report.

| Component | Detail |
|-----------|--------|
| Research model | `o3-deep-research` (Norway East via APIM) |
| Synthesis model | `gpt-4.1-mini` (primary core via APIM) |
| Knowledge base | `arxiv-nlp-kb` (Lab 10 Foundry IQ KB) |
| Corpus | 3,000 NLP research paper abstracts |

## Prerequisites

- Lab 10 complete — `arxiv-nlp-kb` exists, `IQ_SEARCH_ENDPOINT` and `IQ_GATEWAY_KEY` in `.env`
- Lab 12-01 complete (or Lab 05-02) — `DR_MODEL` and `DR_GATEWAY_KEY` in `.env`

## Step 1: Load configuration

In [ ]:
import hashlib
import json
import os
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from openai import AzureOpenAI

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

GATEWAY_URL        = os.environ['GATEWAY_URL']
CHAT_MODEL         = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
IQ_SEARCH_ENDPOINT = os.environ['IQ_SEARCH_ENDPOINT']
IQ_GATEWAY_KEY     = os.environ['IQ_GATEWAY_KEY']
DR_MODEL           = os.environ.get('DR_MODEL', 'o3-deep-research')
DR_GATEWAY_KEY     = os.environ['DR_GATEWAY_KEY']

KB_NAME                = 'arxiv-nlp-kb'
MAX_RESEARCH_ITERATIONS = 10

# AzureOpenAI SDK appends /openai/ to azure_endpoint automatically.
# GATEWAY_URL already ends with /openai — strip it to avoid double-prefix.
apim_base = GATEWAY_URL.rstrip('/').removesuffix('/openai')

print(f'Gateway URL          : {GATEWAY_URL}')
print(f'APIM base            : {apim_base}')
print(f'IQ Search endpoint   : {IQ_SEARCH_ENDPOINT}')
print(f'Chat model           : {CHAT_MODEL}')
print(f'Deep research model  : {DR_MODEL}')
print(f'KB name              : {KB_NAME}')

## Step 2: Initialize Azure clients

In [ ]:
credential = DefaultAzureCredential()

# Deep research client — routes to Norway East research hub via APIM
dr_client = AzureOpenAI(
    azure_endpoint=apim_base,
    api_key=DR_GATEWAY_KEY,
    api_version='2024-12-01-preview',
    timeout=600,    # o3-deep-research can run for several minutes
)

# Chat client — gpt-4.1-mini for final report synthesis
chat_client = AzureOpenAI(
    azure_endpoint=apim_base,
    api_key=IQ_GATEWAY_KEY,
    api_version='2024-10-21',
    timeout=120,
)

print('✅ Deep research client  : ready (o3-deep-research via APIM)')
print('✅ Chat client           : ready (gpt-4.1-mini via APIM)')

## Step 3: Define Foundry IQ client

Queries the `arxiv-nlp-kb` knowledge base via the Foundry IQ retrieve API.
`DefaultAzureCredential` authenticates to Azure AI Search using the caller's
Entra identity — no API keys needed for the search service.

In [ ]:
def _get_search_token() -> str:
    """Get a bearer token scoped to Azure AI Search."""
    return credential.get_token('https://search.azure.com/.default').token


def query_kb(query: str, kb_name: str = KB_NAME) -> dict:
    """Query a Foundry IQ knowledge base using the retrieve API."""
    url = (
        f'{IQ_SEARCH_ENDPOINT.rstrip("/")}/knowledgebases/{kb_name}'
        f'/retrieve?api-version=2025-11-01-preview'
    )
    resp = requests.post(
        url,
        headers={
            'Authorization': f'Bearer {_get_search_token()}',
            'Content-Type': 'application/json',
        },
        json={
            'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': query}]}]
        },
        timeout=60,
    )
    if resp.ok:
        return resp.json()
    return {'error': resp.status_code, 'message': resp.text}


# Smoke-test the KB
test = query_kb('What is few-shot learning?')
if 'error' in test:
    raise RuntimeError(f'KB query failed: {test}. Check IQ_SEARCH_ENDPOINT and that Lab 10 is complete.')
print(f'✅ Foundry IQ KB ({KB_NAME}) reachable')

## Step 4: Define research tools

Two tools are exposed to `o3-deep-research` via function calling:

- **`search`** — queries `arxiv-nlp-kb` and returns summarised results with IDs
- **`fetch`** — retrieves the full cached document content by ID for deeper analysis

In [ ]:
# In-session document cache (populated by search, read by fetch)
_doc_cache: Dict[str, Dict[str, Any]] = {}

TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'search',
            'description': (
                'Search the arxiv-nlp corpus of NLP research papers. '
                'Returns summaries with document IDs that can be fetched for full content.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Natural language search query'}
                },
                'required': ['query'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'fetch',
            'description': (
                'Fetch full content of a document by ID for in-depth reading and citation. '
                'Use after search to get the complete abstract and metadata.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'document_id': {
                        'type': 'string',
                        'description': 'Document ID returned by the search tool'
                    }
                },
                'required': ['document_id'],
            },
        },
    },
]


def tool_search(query: str) -> Dict[str, Any]:
    """Execute the search tool — queries Foundry IQ and returns document summaries."""
    print(f'   🔍 search("{query[:60]}...")')

    result = query_kb(query)
    if 'error' in result:
        return {'error': result.get('message', str(result)), 'results': []}

    documents: List[Dict[str, Any]] = []
    for msg in result.get('response', []):
        for item in msg.get('content', []):
            if item.get('type') == 'text':
                try:
                    docs_json = json.loads(item.get('text', '[]'))
                    if isinstance(docs_json, list):
                        for doc in docs_json[:10]:
                            doc_id = str(
                                doc.get('ref_id')
                                or hashlib.md5(str(doc).encode()).hexdigest()[:12]
                            )
                            parsed = {
                                'id':    doc_id,
                                'title': doc.get('title', 'Untitled'),
                                'text':  doc.get('content', '')[:500] + '...',
                            }
                            documents.append(parsed)
                            _doc_cache[doc_id] = {
                                'id':    doc_id,
                                'title': doc.get('title', 'Untitled'),
                                'text':  doc.get('content', ''),
                            }
                except json.JSONDecodeError:
                    pass

    print(f'      → {len(documents)} document(s) found')
    return {'query': query, 'total_results': len(documents), 'results': documents}


def tool_fetch(document_id: str) -> Dict[str, Any]:
    """Execute the fetch tool — returns cached full document content."""
    print(f'   📄 fetch("{document_id}")')
    if document_id not in _doc_cache:
        return {'error': f'Document "{document_id}" not found. Call search first.'}
    doc = _doc_cache[document_id]
    print(f'      → "{doc["title"][:50]}"')
    return doc


def execute_tool(name: str, arguments: Dict[str, Any]) -> str:
    """Dispatch a tool call and return the JSON result as a string."""
    if name == 'search':
        result = tool_search(arguments.get('query', ''))
    elif name == 'fetch':
        result = tool_fetch(arguments.get('document_id', ''))
    else:
        result = {'error': f'Unknown tool: {name}'}
    return json.dumps(result, indent=2)


print('✅ Research tools defined')
print('   - search : query Foundry IQ arxiv-nlp-kb')
print('   - fetch  : retrieve full document content by ID')

## Step 5: Define the deep research runner

The agentic loop:
1. Sends the query to `o3-deep-research` with tool definitions
2. Executes any tool calls against Foundry IQ, appends results to the message chain
3. Repeats until the model returns a message with no tool calls
4. Passes the model's reasoning to `gpt-4.1-mini` for final synthesis and formatting

In [ ]:
@dataclass
class ResearchResult:
    query: str
    iterations: int = 0
    tool_calls: List[Dict[str, Any]] = field(default_factory=list)
    final_answer: str = ''
    reasoning_tokens: int = 0
    total_tokens: int = 0
    duration_seconds: float = 0.0
    error: Optional[str] = None


def run_deep_research(query: str) -> ResearchResult:
    """Run the agentic deep research loop and return a ResearchResult."""
    result = ResearchResult(query=query)
    start_time = time.time()

    system_prompt = (
        'You are a deep research assistant with access to a corpus of NLP research papers '
        '(arXiv abstracts). Your task is to thoroughly research the user\'s query by:\n'
        '1. Using the "search" tool to find relevant papers in the knowledge base\n'
        '2. Using the "fetch" tool to retrieve full content of the most relevant papers\n'
        '3. Analysing and synthesising information across multiple papers\n'
        '4. Providing a comprehensive, well-cited answer\n\n'
        'IMPORTANT:\n'
        '- Search multiple times with different query formulations for comprehensive coverage\n'
        '- Fetch papers that look directly relevant before writing your final answer\n'
        '- Include specific technical details, model names, benchmarks, and results from papers\n'
        '- Cite sources using document IDs (e.g. [doc-abc123])\n'
        '- Structure your final answer with clear sections and headers\n'
        '- If a query asks about something outside the NLP corpus, say so explicitly'
    )

    messages: List[Dict[str, Any]] = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': query},
    ]

    print(f'\n{"="*65}')
    print('🔬 DEEP RESEARCH STARTED')
    print(f'{"="*65}')
    print(f'Query: {query[:100]}...' if len(query) > 100 else f'Query: {query}')
    print()

    try:
        for iteration in range(MAX_RESEARCH_ITERATIONS):
            print(f'📍 Iteration {iteration + 1}/{MAX_RESEARCH_ITERATIONS}')

            response = dr_client.chat.completions.create(
                model=DR_MODEL,
                messages=messages,
                tools=TOOLS,
            )

            message = response.choices[0].message
            if response.usage:
                result.total_tokens += response.usage.total_tokens
                details = getattr(response.usage, 'completion_tokens_details', None)
                if details:
                    result.reasoning_tokens += getattr(details, 'reasoning_tokens', 0) or 0

            # No tool calls — research phase complete; synthesise with gpt-4.1-mini
            if not message.tool_calls:
                print('\n✅ Research phase complete — synthesising final report with gpt-4.1-mini...')
                research_context = message.content or ''

                synthesis = chat_client.chat.completions.create(
                    model=CHAT_MODEL,
                    messages=[
                        {
                            'role': 'system',
                            'content': (
                                'You are an expert research report writer. '
                                'Synthesise the deep research findings into a structured, '
                                'well-formatted report. Use clear Markdown headers. '
                                'Preserve all citations and technical details. '
                                'Be thorough and precise.'
                            ),
                        },
                        {
                            'role': 'user',
                            'content': (
                                f'Original query:\n{query}\n\n'
                                f'Research findings:\n{research_context}\n\n'
                                'Write a comprehensive, well-organised research report.'
                            ),
                        },
                    ],
                )
                result.final_answer = synthesis.choices[0].message.content or ''
                if synthesis.usage:
                    result.total_tokens += synthesis.usage.total_tokens
                result.iterations = iteration + 1
                break

            # Process tool calls
            messages.append(message)
            for tool_call in message.tool_calls[:5]:  # guard against runaway loops
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                tool_result = execute_tool(func_name, func_args)
                result.tool_calls.append({
                    'iteration': iteration + 1,
                    'tool': func_name,
                    'arguments': func_args,
                })
                messages.append({
                    'role': 'tool',
                    'tool_call_id': tool_call.id,
                    'content': tool_result,
                })

            # Max iterations reached — synthesise from gathered tool results
            if iteration == MAX_RESEARCH_ITERATIONS - 1:
                print('\n⚠️  Max iterations reached — synthesising from gathered data...')
                gathered = [
                    msg['content']
                    for msg in messages
                    if isinstance(msg, dict) and msg.get('role') == 'tool'
                ]
                synthesis = chat_client.chat.completions.create(
                    model=CHAT_MODEL,
                    messages=[
                        {
                            'role': 'system',
                            'content': (
                                'You are an expert research report writer. '
                                'Synthesise the gathered research data into a structured report.'
                            ),
                        },
                        {
                            'role': 'user',
                            'content': (
                                f'Original query:\n{query}\n\n'
                                f'Gathered research data:\n{chr(10).join(gathered[:8])}\n\n'
                                'Write a comprehensive, well-organised research report.'
                            ),
                        },
                    ],
                )
                result.final_answer = synthesis.choices[0].message.content or ''
                if synthesis.usage:
                    result.total_tokens += synthesis.usage.total_tokens
                result.iterations = iteration + 1

    except Exception as exc:
        result.error = str(exc)
        print(f'\n❌ Error: {exc}')

    result.duration_seconds = round(time.time() - start_time, 2)

    print(f'\n{"="*65}')
    print('🔬 DEEP RESEARCH COMPLETE')
    print(f'{"="*65}')
    print(f'   Iterations       : {result.iterations}')
    print(f'   Tool calls       : {len(result.tool_calls)}')
    print(f'   Total tokens     : {result.total_tokens:,}')
    print(f'   Reasoning tokens : {result.reasoning_tokens:,}')
    print(f'   Duration         : {result.duration_seconds}s')

    return result


def display_result(res: ResearchResult) -> None:
    """Render a ResearchResult with a summary card and the final report."""
    if res.error:
        display(HTML(
            f'<div style="background:#ffdddd;padding:15px;border-radius:8px;">'
            f'<h3>❌ Research Error</h3><p>{res.error}</p></div>'
        ))
        return

    card = f'''
    <div style="font-family:system-ui;padding:20px;background:linear-gradient(135deg,#1a1a2e,#16213e);
                border-radius:12px;margin:10px 0;">
      <h2 style="color:#4da6ff;margin:0 0 15px 0;">🔬 Deep Research Results</h2>
      <p style="color:#ccc;margin:0 0 15px 0;font-size:14px;">{res.query[:120]}{'...' if len(res.query)>120 else ''}</p>
      <div style="display:flex;gap:16px;flex-wrap:wrap;">
        <div style="background:rgba(15,52,96,0.4);padding:12px 20px;border-radius:8px;text-align:center;">
          <div style="font-size:24px;color:#4da6ff;font-weight:bold;">{res.iterations}</div>
          <div style="color:#888;font-size:12px;">Iterations</div>
        </div>
        <div style="background:rgba(15,52,96,0.4);padding:12px 20px;border-radius:8px;text-align:center;">
          <div style="font-size:24px;color:#28a745;font-weight:bold;">{len(res.tool_calls)}</div>
          <div style="color:#888;font-size:12px;">Tool Calls</div>
        </div>
        <div style="background:rgba(15,52,96,0.4);padding:12px 20px;border-radius:8px;text-align:center;">
          <div style="font-size:24px;color:#ffc107;font-weight:bold;">{res.total_tokens:,}</div>
          <div style="color:#888;font-size:12px;">Total Tokens</div>
        </div>
        <div style="background:rgba(15,52,96,0.4);padding:12px 20px;border-radius:8px;text-align:center;">
          <div style="font-size:24px;color:#e94560;font-weight:bold;">{res.duration_seconds}s</div>
          <div style="color:#888;font-size:12px;">Duration</div>
        </div>
        <div style="background:rgba(15,52,96,0.4);padding:12px 20px;border-radius:8px;text-align:center;">
          <div style="font-size:24px;color:#9b59b6;font-weight:bold;">{res.reasoning_tokens:,}</div>
          <div style="color:#888;font-size:12px;">Reasoning Tokens</div>
        </div>
      </div>
    </div>
    '''
    display(HTML(card))

    if res.tool_calls:
        tc_html = '<details style="margin:8px 0;"><summary style="cursor:pointer;color:#888;">'\
                  f'Tool calls ({len(res.tool_calls)})</summary><pre style="font-size:12px;'\
                  'background:#111;color:#ccc;padding:10px;border-radius:4px;overflow:auto;">'
        for tc in res.tool_calls:
            tc_html += f"[iter {tc['iteration']}] {tc['tool']}({json.dumps(tc['arguments'])})"
            if tc is not res.tool_calls[-1]:
                tc_html += '\n'
        tc_html += '</pre></details>'
        display(HTML(tc_html))

    display(Markdown(res.final_answer))


print('✅ Deep research runner ready')

## Step 6: Research Query 1 — Few-shot learning approaches

> *What are the main approaches to few-shot learning described in the corpus?
> Compare their performance characteristics and typical benchmarks.*

In [ ]:
_doc_cache.clear()

query_1 = (
    'What are the main approaches to few-shot learning described in the corpus? '
    'Compare their performance characteristics, the benchmarks they are evaluated on, '
    'and what makes each approach distinct. Include specific model names and reported results.'
)

result_1 = run_deep_research(query_1)
display_result(result_1)

## Step 7: Research Query 2 — Transformer attention efficiency

> *Summarise the evolution of transformer attention mechanisms across the papers.
> What are the key efficiency improvements cited?*

In [ ]:
_doc_cache.clear()

query_2 = (
    'Summarise the evolution of transformer attention mechanisms described across papers in the corpus. '
    'What are the key efficiency improvements? How do they address the quadratic complexity of '
    'standard self-attention? Include specific technique names, complexity bounds, and results '
    'where available.'
)

result_2 = run_deep_research(query_2)
display_result(result_2)

## Step 8: Research Query 3 — Multilingual NLP

> *Which papers address multilingual NLP? What languages and tasks are covered?*

In [ ]:
_doc_cache.clear()

query_3 = (
    'Which papers in the corpus address multilingual NLP? '
    'What languages are covered and what NLP tasks are evaluated (translation, NER, QA, etc.)? '
    'Identify any cross-lingual transfer learning approaches and how they handle low-resource languages.'
)

result_3 = run_deep_research(query_3)
display_result(result_3)

## Step 9: Research Query 4 — Out-of-scope boundary demonstration

> *What are the latest breakthroughs in nuclear fusion energy research?*
>
> This query is deliberately outside the NLP corpus. It demonstrates that the model
> correctly identifies the knowledge boundary and does not hallucinate an answer.

In [ ]:
_doc_cache.clear()

query_4 = (
    'What are the latest breakthroughs in nuclear fusion energy research? '
    'Summarise the most recent achievements, the institutions involved, and '
    'the challenges still to be overcome.'
)

result_4 = run_deep_research(query_4)
display_result(result_4)